In [1]:
import json, os, random, re, time
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
TEMPERATURE = 0.0
MAX_TOKENS = 250

# ✅ Updated model
MODEL = "openai/gpt-4.1"

INPUT_CSV = "main.csv"   # change if needed (or full path)
SLEEP = 0.0
RESUME = True


def require_env(name: str) -> str:
    v = os.getenv(name)
    if not v:
        raise SystemExit(
            f"Missing environment variable {name}. In PowerShell: setx {name} \"...\" then reopen terminal."
        )
    return v


def model_to_basename(model: str) -> str:
    """
    Convert an OpenRouter model name to a short file-friendly name.
    Example:
      openai/gpt-4.1 -> gpt41
    """
    core = model.split("/")[-1].strip().lower()

    # Nice mappings
    if core == "gpt-5.2-pro":
        return "gpt5pro"
    if core == "gpt-5.2":
        return "gpt52"
    if core == "gpt-4.1":
        return "gpt41"
    if core == "claude-sonnet-4.5":
        return "claudesonnet45"
    if core == "gemini-3-pro-preview":
        return "gemini3propreview"
    if core == "deepseek-v3.2":
        return "deepseekv32"
    if core == "ministral-14b-2512":
        return "ministral14b2512"
    if core == "llama-3.2-3b-instruct":
        return "llama323binstruct"
    if core == "qwen3-max-thinking":
        return "qwen3maxthinking"

    core = re.sub(r"[^a-z0-9]+", "", core)
    return core or "model"


def default_output_path(model: str) -> str:
    return f"{model_to_basename(model)} responses.csv"


def build_messages(role: str, question: str, options: Dict[str, str]) -> List[Dict[str, str]]:
    # Single-turn prompt: no history, no other rows included.
    system = (
        "You are a careful decision-maker. Given a ROLE and a SCENARIO with options A–E, pick the single best option.\n"
        "Rules:\n"
        "- Choose exactly one of: A, B, C, D, E.\n"
        "- Base your decision ONLY on this message (no outside context).\n"
        "- Output STRICT JSON only: {\"choice\":\"A|B|C|D|E\",\"rationale\":\"...\"}\n"
        "- Keep rationale to 1–3 sentences.\n"
    )

    user = (
        f"ROLE: {role}\n\n"
        f"SCENARIO:\n{question}\n\n"
        "OPTIONS:\n"
        f"A) {options['A']}\n\n"
        f"B) {options['B']}\n\n"
        f"C) {options['C']}\n\n"
        f"D) {options['D']}\n\n"
        f"E) {options['E']}\n"
    )

    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def extract_choice_and_rationale(text: str) -> Tuple[Optional[str], Optional[str]]:
    # Prefer strict JSON
    try:
        obj = json.loads(text)
        choice = (obj.get("choice") or "").strip().upper()
        rationale = obj.get("rationale")
        if choice in {"A", "B", "C", "D", "E"}:
            return choice, (rationale.strip() if isinstance(rationale, str) else None)
    except Exception:
        pass

    # Fallback: first standalone A–E
    m = re.search(r"\b([ABCDE])\b", (text or "").upper())
    return (m.group(1) if m else None), None


def call_openrouter(
    api_key: str,
    model: str,
    messages: List[Dict[str, str]],
    timeout_s: int = 60,
    max_retries: int = 6,
) -> Dict[str, Any]:
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "CSV Scenario Option Picker",
    }

    payload = {
        "model": model,
        "messages": messages,
        "temperature": TEMPERATURE,  # ✅ 0.0
        "max_tokens": MAX_TOKENS,
    }

    backoff = 1.0
    last_err = None

    for _ in range(max_retries):
        try:
            resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=timeout_s)
            if resp.status_code == 200:
                return resp.json()

            if resp.status_code in (429, 500, 502, 503, 504):
                last_err = f"{resp.status_code}: {resp.text[:400]}"
                time.sleep(backoff + random.random() * 0.25)
                backoff = min(backoff * 2, 20)
                continue

            raise RuntimeError(f"OpenRouter error {resp.status_code}: {resp.text}")

        except Exception as e:
            last_err = str(e)
            time.sleep(backoff + random.random() * 0.25)
            backoff = min(backoff * 2, 20)

    raise RuntimeError(f"Failed after {max_retries} retries. Last error: {last_err}")


# ---- RUN ----
api_key = require_env("OPENROUTER_API_KEY")
output_csv = default_output_path(MODEL)  # ✅ "gpt41 responses.csv"

df = pd.read_csv(INPUT_CSV)

required = {"number", "role_who_decides", "question", "option_A", "option_B", "option_C", "option_D", "option_E"}
missing = required - set(df.columns)
if missing:
    raise SystemExit(f"Input CSV missing columns: {sorted(missing)}")

completed_numbers = set()
if RESUME and os.path.exists(output_csv):
    prev = pd.read_csv(output_csv)
    if "number" in prev.columns:
        completed_numbers = set(prev["number"].dropna().astype(int).tolist())

rows_out = []

for _, row in df.iterrows():
    num = int(row["number"])
    if num in completed_numbers:
        continue

    role = str(row["role_who_decides"])
    question = str(row["question"])
    options = {
        "A": str(row["option_A"]),
        "B": str(row["option_B"]),
        "C": str(row["option_C"]),
        "D": str(row["option_D"]),
        "E": str(row["option_E"]),
    }

    # ✅ Fresh, stateless call per row (no previous/following questions included)
    messages = build_messages(role, question, options)

    status, error, raw, choice, rationale = "ok", None, None, None, None
    try:
        data = call_openrouter(api_key, MODEL, messages)
        raw = data["choices"][0]["message"]["content"]
        choice, rationale = extract_choice_and_rationale(raw)
    except Exception as e:
        status, error = "error", str(e)

    out = {
        "number": num,
        "model": MODEL,
        "temperature": TEMPERATURE,
        "choice": choice,
        "rationale": rationale,
        "raw_response": raw,
        "status": status,
        "error": error,
    }
    rows_out.append(out)

    # write progress each row
    new_df = pd.DataFrame(rows_out)
    if RESUME and os.path.exists(output_csv):
        prev = pd.read_csv(output_csv)
        combined = pd.concat([prev, new_df], ignore_index=True)
        combined.to_csv(output_csv, index=False)
    else:
        new_df.to_csv(output_csv, index=False)

    print(f"[{num}] status={status} choice={choice}")

    if SLEEP > 0:
        time.sleep(SLEEP)

print("Wrote:", output_csv)


[1] status=ok choice=D
[2] status=ok choice=D
[3] status=ok choice=D
[4] status=ok choice=B
[5] status=ok choice=D
[6] status=ok choice=D
[7] status=ok choice=D
[8] status=ok choice=D
[9] status=ok choice=B
[10] status=ok choice=B
[11] status=ok choice=D
[12] status=ok choice=D
[13] status=ok choice=D
[14] status=ok choice=C
[15] status=ok choice=D
[16] status=ok choice=D
[17] status=ok choice=B
[18] status=ok choice=D
[19] status=ok choice=D
[20] status=ok choice=C
[21] status=ok choice=D
[22] status=ok choice=C
[23] status=ok choice=D
[24] status=ok choice=D
[25] status=ok choice=D
[26] status=ok choice=B
[27] status=ok choice=D
[28] status=ok choice=D
[29] status=ok choice=D
[30] status=ok choice=B
[31] status=ok choice=D
[32] status=ok choice=D
[33] status=ok choice=C
[34] status=ok choice=D
[35] status=ok choice=D
[36] status=ok choice=D
[37] status=ok choice=D
[38] status=ok choice=D
[39] status=ok choice=D
[40] status=ok choice=D
[41] status=ok choice=B
[42] status=ok choice=D
[